In [ ]:
############################################Data Preparation (Data Cleaning)############################################
import pandas as pd
import os

############################################ 1 - Load the original dataset

df = pd.read_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025.csv",sep=',' , encoding="utf-8")
print(df.columns)
print(df.head(5))
print(df.tail(5))
print(df.shape)

In [ ]:
############################################ 2- Checking corrupted rows
import re

corrupted_rows = []
df['loan_status'] = df['loan_status'].astype(int)

# Business rule validations
for idx, row in df.iterrows():
    issues = []
    
    # Age validation (18-100)
    if pd.notna(row['age']) and (row['age'] < 18 or row['age'] > 100):
        issues.append('Invalid age')
    
    # Credit score validation (300-850)
    if pd.notna(row['credit_score']) and (row['credit_score'] < 300 or row['credit_score'] > 850):
        issues.append('Invalid credit_score')
    
    # Years employed validation (0-60)
    if pd.notna(row['years_employed']) and (row['years_employed'] < 0 or row['years_employed'] > 60):
        issues.append('Invalid years_employed')
    
    # Credit history years validation (0-80)
    if pd.notna(row['credit_history_years']) and (row['credit_history_years'] < 0 or row['credit_history_years'] > 80):
        issues.append('Invalid credit_history_years')
    
    # Negative values validation
    if pd.notna(row['annual_income']) and row['annual_income'] < 0:
        issues.append('Negative annual_income')
    if pd.notna(row['savings_assets']) and row['savings_assets'] < 0:
        issues.append('Negative savings_assets')
    if pd.notna(row['current_debt']) and row['current_debt'] < 0:
        issues.append('Negative current_debt')
    if pd.notna(row['loan_amount']) and row['loan_amount'] < 0:
        issues.append('Negative loan_amount')
    
    # Interest rate validation (0-100%)
    if pd.notna(row['interest_rate']) and (row['interest_rate'] < 0 or row['interest_rate'] > 100):
        issues.append('Invalid interest_rate')
    
    # Ratio validations (0-100)
    if pd.notna(row['debt_to_income_ratio']) and (row['debt_to_income_ratio'] < 0 or row['debt_to_income_ratio'] > 100):
        issues.append('Invalid debt_to_income_ratio')
    if pd.notna(row['loan_to_income_ratio']) and (row['loan_to_income_ratio'] < 0 or row['loan_to_income_ratio'] > 100):
        issues.append('Invalid loan_to_income_ratio')
    if pd.notna(row['payment_to_income_ratio']) and (row['payment_to_income_ratio'] < 0 or row['payment_to_income_ratio'] > 100):
        issues.append('Invalid payment_to_income_ratio')
    
    # Defaults and delinquencies validation (non-negative)
    if pd.notna(row['defaults_on_file']) and row['defaults_on_file'] < 0:
        issues.append('Negative defaults_on_file')
    if pd.notna(row['delinquencies_last_2yrs']) and row['delinquencies_last_2yrs'] < 0:
        issues.append('Negative delinquencies_last_2yrs')
    if pd.notna(row['derogatory_marks']) and row['derogatory_marks'] < 0:
        issues.append('Negative derogatory_marks')
    
    # String field validation using regex
    if pd.notna(row['occupation_status']):
        if not re.match(r'^[A-Za-z\s\-]+$', str(row['occupation_status'])):
            issues.append('Malformed occupation_status')
    
    if pd.notna(row['product_type']):
        if not re.match(r'^[A-Za-z\s\-]+$', str(row['product_type'])):
            issues.append('Malformed product_type')
    
    if pd.notna(row['loan_status']):
       if int(row['loan_status']) not in [0, 1]:
           issues.append('Malformed loan_status')

    
    # Logical inconsistencies
    if pd.notna(row['credit_history_years']) and pd.notna(row['age']):
        if row['credit_history_years'] > row['age'] - 18:
            issues.append('Credit history exceeds possible years')
    
    if pd.notna(row['years_employed']) and pd.notna(row['age']):
        if row['years_employed'] > row['age'] - 18:
            issues.append('Years employed exceeds possible years')
    
    # Missing critical values
    if pd.isna(row['customer_id']) or pd.isna(row['loan_status']):
        issues.append('Missing critical field')
    
    if issues:
        corrupted_rows.append({
            'index': idx,
            'customer_id': row['customer_id'],
            'issues': ', '.join(issues),
            'row_data': row
        })

# Display results
print(f"Total corrupted rows found: {len(corrupted_rows)}\n")

for corrupt in corrupted_rows:
    print(f"Row Index: {corrupt['index']}")
    print(f"Customer ID: {corrupt['customer_id']}")
    print(f"Issues: {corrupt['issues']}")
    print(f"Data: {corrupt['row_data'].to_dict()}")
    print("-" * 80)

#################### we found 144 rows that loan_to_income_ratio & debt_to_income_ratio have inproper ratio so:

# Remove rows where annual_income is missing or zero
df = df[df['annual_income'].notna()]
df = df[df['annual_income'] > 0]

# Recalculate ratios
df['loan_to_income_ratio'] = df['loan_amount'] / df['annual_income']
df['debt_to_income_ratio'] = df['current_debt'] / df['annual_income']

# Remove first row if needed
df_corrected = df.iloc[1:].copy()


# Save dataset
df_corrected.to_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025-1.csv", index=False)

print("✅ New corrected dataset saved as Loan_approval_data_2025-1.csv")

df = pd.read_csv(r"D:\Thesis\4-Dataset\Loan_approval_data_2025-1.csv")

In [ ]:
############################################ 3-Checking Column Types Fix column data types
df["age"] = df["age"].astype(int)
df["defaults_on_file"] = df["defaults_on_file"].astype(int)
df["delinquencies_last_2yrs"] = df["delinquencies_last_2yrs"].astype(int)
df["derogatory_marks"] = df["derogatory_marks"].astype(int)
df["loan_status"] = df["loan_status"].astype(int)

############################################ 4-handle missing value
print(df.isna().sum(axis=1))

############################################ 5-normalize inconsistent text (“Male” vs “male”) in rows
print(df["occupation_status"].value_counts())
print(df["product_type"].value_counts())
print(df["loan_intent"].value_counts())

In [ ]:
############################################ 6- recognizing missing value
print(df.isna().sum())

############################################ 7- Preprocessing: Handle missing values if necessary
df.dropna(inplace=True)      # Example: remove rows with missing values

 ############################################ 8-Cheking Duplicated row
duplicates_count = df.duplicated().sum()
# Check if there are any duplicate rows and print the result using f-strings
if df.duplicated().any():
    print(f"Duplicates are present. Total duplicate rows: {duplicates_count}")
    df = df.drop_duplicates()
    print("Duplicates values is deleted")
else:
    print(f"No duplicates are present in the Dataset.")
